# Assignment 2



In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
import pandas as pd
import numpy as np
import os
import sys
import dask.dataframe as dd
#sys.path.append("../../05_src")

c:\Users\fansh\anaconda4\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [ ]:
target_column = 'area'

In [4]:
# Features (X): All columns except the target
X = fires_dt.drop(columns=[target_column])

# Target (y): The column to predict
y = fires_dt[target_column]

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [5]:
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer

# Define feature types
num_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
cat_features = ['month', 'day']

# Log transformation for certain numerical features
log_transformer = FunctionTransformer(np.log1p, validate=True)  # log(1 + x) to handle zeros

### Preprocessing 1: Standard Scaling + One-Hot Encoding
preproc1 = ColumnTransformer([
    ('num_scaler', StandardScaler(), num_features),  # Standard Scaler for numerical features
    ('cat_encoder', OneHotEncoder(handle_unknown='ignore'), cat_features)  # One-Hot Encoding
])


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [6]:
preproc2 = ColumnTransformer([
    ('num_log_scaler', RobustScaler(), num_features),  # Robust Scaler for numerical features
    ('num_log_transform', log_transformer, ['ffmc', 'isi']),  # Log Transform specific features
    ('cat_encoder', OneHotEncoder(handle_unknown='ignore'), cat_features)  # One-Hot Encoding
])


In [7]:
# Fit these transformers on the data set
X_preproc1 = preproc1.fit_transform(fires_dt.drop(columns=['area']))
X_preproc2 = preproc2.fit_transform(fires_dt.drop(columns=['area']))

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score

In [11]:
# Baseline Model: Linear Regression
baseline = LinearRegression()
pipelines = {
    "Pipeline A": Pipeline([('preprocessing', preproc1), ('regressor', baseline)])
}

In [12]:
# Advanced Model: Random Forest Regressor
advanced_model = RandomForestRegressor(random_state=42)

In [13]:
# Pipeline A = preproc1 + baseline
# Pipeline B = preproc2 + baseline
# Pipeline C = preproc1 + advanced model
# Pipeline D = preproc2 + advanced model
pipelines = {
    "Pipeline A": Pipeline([('preprocessing', preproc1), ('regressor', baseline)]),
    "Pipeline B": Pipeline([('preprocessing', preproc2), ('regressor', baseline)]),
    "Pipeline C": Pipeline([('preprocessing', preproc1), ('regressor', advanced_model)]),
    "Pipeline D": Pipeline([('preprocessing', preproc2), ('regressor', advanced_model)])
}

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [ ]:
# Extract Features (X) and Target (y)
X = fires_dt.drop(columns=['area'])  # Features
y = fires_dt['area']  # Target
# Evaluate Each Pipeline using Cross-Validation
for name, pipeline in pipelines.items():
    score = cross_val_score(pipeline, X, y, cv=5, scoring='r2').mean()
    print(f'{name} R² Score: {score:.4f}')

# Grid Search for Advanced Models (Pipelines C & D)
param_grid = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5, 10]
}

for name in ["Pipeline C", "Pipeline D"]:
    grid_search = GridSearchCV(pipelines[name], param_grid, cv=5, scoring='r2', n_jobs=-1)
    grid_search.fit(X, y)
    print(f'Best Parameters for {name}: {grid_search.best_params_}')
    print(f'Best {name} R² Score: {grid_search.best_score_:.4f}')

In [14]:
# Define hyperparameter grids
param_grids = {
    "Pipeline A": {
        'regressor__fit_intercept': [True, False],
    },
    "Pipeline B": {
        'regressor__fit_intercept': [True, False],
    },
    "Pipeline C": {
        'regressor__n_estimators': [50, 100, 200, 300],  # Number of trees
        'regressor__max_depth': [None, 10, 20, 30],  # Depth of trees
        'regressor__min_samples_split': [2, 5, 10],  # Minimum samples per split
    },
    "Pipeline D": {
        'regressor__n_estimators': [50, 100, 200, 300],  
        'regressor__max_depth': [None, 10, 20, 30],  
        'regressor__min_samples_split': [2, 5, 10],  
    }
}

In [15]:
# Extract Features (X) and Target (y)
X = fires_dt.drop(columns=['area'])  # Features
y = fires_dt['area']  # Target

# Perform Grid Search for each pipeline
best_params = {}
best_scores = {}

for name, pipeline in pipelines.items():
    print(f"\n🔍 Tuning {name}...")

    grid_search = GridSearchCV(pipeline, param_grids[name], cv=5, scoring='r2', n_jobs=-1, verbose=1)
    grid_search.fit(X, y)

    # Store best parameters and best score
    best_params[name] = grid_search.best_params_
    best_scores[name] = grid_search.best_score_

    print(f"✅ Best Params for {name}: {grid_search.best_params_}")
    print(f"🏆 Best R² Score for {name}: {grid_search.best_score_:.4f}")

# Display final results
print("\n🎯 Final Best Parameters and Scores:")
for name in pipelines.keys():
    print(f"{name}: Best R² Score = {best_scores[name]:.4f}, Best Params = {best_params[name]}")


🔍 Tuning Pipeline A...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
✅ Best Params for Pipeline A: {'regressor__fit_intercept': True}
🏆 Best R² Score for Pipeline A: -4.6181

🔍 Tuning Pipeline B...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
✅ Best Params for Pipeline B: {'regressor__fit_intercept': True}
🏆 Best R² Score for Pipeline B: -8.4029

🔍 Tuning Pipeline C...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
✅ Best Params for Pipeline C: {'regressor__max_depth': None, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 50}
🏆 Best R² Score for Pipeline C: -3.3928

🔍 Tuning Pipeline D...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
✅ Best Params for Pipeline D: {'regressor__max_depth': None, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 50}
🏆 Best R² Score for Pipeline D: -4.3524

🎯 Final Best Parameters and Scores:
Pipeline A: Best R² Score = -4.6181, Best Params = {'regressor__fit_intercept

# Evaluate

+ Which model has the best performance?

In [16]:
# Print final results
print("\n🎯 Final Best Parameters and Scores:")

# Track the best pipeline
best_pipeline = None
best_r2_score = -float("inf")  # Initialize with a very low score

for name in pipelines.keys():
    print(f"{name}: Best R² Score = {best_scores[name]:.4f}, Best Params = {best_params[name]}")
    
    # Find the best performing pipeline
    if best_scores[name] > best_r2_score:
        best_r2_score = best_scores[name]
        best_pipeline = name

# Print the best model
print("\n🏆 The Best Performing Model is:")
print(f"🚀 {best_pipeline} with R² Score = {best_r2_score:.4f}")



🎯 Final Best Parameters and Scores:
Pipeline A: Best R² Score = -4.6181, Best Params = {'regressor__fit_intercept': True}
Pipeline B: Best R² Score = -8.4029, Best Params = {'regressor__fit_intercept': True}
Pipeline C: Best R² Score = -3.3928, Best Params = {'regressor__max_depth': None, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 50}
Pipeline D: Best R² Score = -4.3524, Best Params = {'regressor__max_depth': None, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 50}

🏆 The Best Performing Model is:
🚀 Pipeline C with R² Score = -3.3928


# Export

+ Save the best performing model to a pickle file.

In [17]:
import pickle

# Train Pipeline C on the full dataset before saving (optional but recommended)
best_model = pipelines["Pipeline C"].fit(X, y)

# Save the trained model
filename = "best_fire_model_pipeline_C.pkl"
with open(filename, "wb") as file:
    pickle.dump(best_model, file)

print(f"\n✅ Pipeline C (Best Model) saved as '{filename}'!")


✅ Pipeline C (Best Model) saved as 'best_fire_model_pipeline_C.pkl'!


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Load the saved best model (Pipeline C)
with open("best_fire_model_pipeline_C.pkl", "rb") as file:
    best_model = pickle.load(file)

# Split dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Explain the model’s predictions using SHAP
explainer = shap.TreeExplainer(best_model.named_steps['regressor'])  # For Random Forest
shap_values = explainer.shap_values(X_test)

# Select a single test instance to explain
idx = 10  # Example row index (can change)
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[idx], X_test.iloc[idx])


*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.